In [1]:
from collections import defaultdict
import math

def affine_char_dict(cartan_type, level, max_ord=15):
    r'''
    Input:
      - cartan_type: Cartan type string (e.g., "A1", "A4") or CartanType object
      - level: positive integer (level k)
      - max_ord: maximum expansion order for q
    Output:
      - Dictionary mapping highest weight vectors to their non-specialised characters
    '''
    ct = CartanType(cartan_type)
    if ct.is_affine():
        ct = ct.classical()

    cartan_mat = ct.cartan_matrix()
    inv_cartan_mat = cartan_mat.inverse()

    d = vector(QQ, ct.symmetrizer())
    d /= max(d)

    G = diagonal_matrix(QQ, d) * cartan_mat
    quadF = inv_cartan_mat.transpose() * G * inv_cartan_mat

    ct_aff = ct.affine()
    index = list(ct_aff.index_set())
    WL = RootSystem(ct_aff).weight_lattice(extended=True)
    Lambda = WL.fundamental_weights()
    rnk = len(index) - 1

    # Simple coroots in fundamental weight coordinates
    C = cartan_mat * diagonal_matrix(QQ, [1/d[i] for i in range(rnk)])
    M = level * C
    inv_M = M.inverse()
    inv_C = C.inverse()

    A_mat = M.transpose() * quadF * M
    
    # Calculate Cholesky decomposition of A_mat via double precision floats for swift sphere traversal
    A_real = A_mat.change_ring(RDF)
    V_cholesky = A_real.cholesky().transpose()

    var_names = ['x%d' % (i + 1) for i in range(rnk)]
    Rx = LaurentPolynomialRing(ZZ, var_names)
    P = PuiseuxSeriesRing(Rx, 'q')
    q = P.gen()

    qF_gcd = 1 / gcd( quadF.list() )
    ram_ord = 2 * level * qF_gcd
    R2 = 2 * level * max_ord

    theta_cache = {}

    def get_points(u0, R2_val):
        """Finds all u in Z^rnk such that (u-u0)*A_mat*(u-u0) <= R2_val via Fincke-Pohst."""
        u0_real = [float(x) for x in u0]
        pts = []
        
        def search_sphere(k, current_R2, partial_sums, current_u):
            if k == -1:
                pts.append(tuple(current_u))
                return
                
            Vkk = V_cholesky[k, k]
            inv_Vkk = 1.0 / Vkk
            S = partial_sums[k]
            # Safety epsilon allows buffer for minute float rounding discrepancies
            bound = math.sqrt(max(0.0, current_R2)) + 1e-9
            
            min_y_k = (-bound - S) * inv_Vkk
            max_y_k = (bound - S) * inv_Vkk
            
            min_u_k = math.ceil(min_y_k + u0_real[k])
            max_u_k = math.floor(max_y_k + u0_real[k])
            
            for uk in range(int(min_u_k), int(max_u_k) + 1):
                yk = uk - u0_real[k]
                term = Vkk * yk + S
                new_R2 = current_R2 - term * term
                
                next_partial = list(partial_sums)
                for i in range(k):
                    next_partial[i] += V_cholesky[i, k] * yk
                
                # FIX: Explicitly assign coordinate uk for this branch
                next_u = list(current_u)
                next_u[k] = uk
                
                search_sphere(k - 1, new_R2, next_partial, next_u)
                
        search_sphere(rnk - 1, float(R2_val), [0.0] * rnk, [0] * rnk)
        
        # Exact filtering using rationals guarantees 100% mathematical precision
        u0_exact = vector(QQ, u0)
        valid_pts = []
        for pt in pts:
            u = vector(QQ, pt)
            du = u - u0_exact
            if du * A_mat * du <= R2_val:
                valid_pts.append(u)
        return valid_pts

    def Theta(v_lamb):
        v_key = tuple(v_lamb)
        if v_key in theta_cache:
            return theta_cache[v_key]

        u0 = - (1 / level) * inv_C * v_lamb
        
        # Native python dictionary tracks occurrences rapidly without object instantiation
        dictseries = defaultdict(lambda: defaultdict(int))
        
        valid_pts = get_points(u0, R2)
        
        for u in valid_pts:
            vec = M * u + v_lamb
            exponent = int(qF_gcd * (vec * quadF * vec))
            mono_key = int(vec[0]) if rnk == 1 else tuple(int(v) for v in vec)
            dictseries[exponent][mono_key] += 1

        # Generates polynomial ring elements purely as a batch operation at loop's end
        poly_dict = {exp: Rx(dict(monos)) for exp, monos in dictseries.items()}
        res = P(poly_dict, e=ram_ord).add_bigoh(max_ord)
        
        theta_cache[v_key] = res
        return res

    def get_classical_orbit(v_start):
        '''Generates the exact classical Weyl group orbit of a weight vector modulo kM'''
        v_start = vector(QQ, v_start)
        orbit = [v_start]
        queue = [v_start]
        
        # Extracts vector representation's fractional remainder to function as a hashed state
        def to_canon(v):
            coords = inv_M * v
            return tuple((c - floor(c)) for c in coords)
            
        visited = {to_canon(v_start)}
        
        while queue:
            curr = queue.pop(0)
            
            for i in range(rnk):
                nxt = curr - curr[i] * cartan_mat.column(i)
                canon_nxt = to_canon(nxt)
                
                # Modulo uniqueness checked in O(1)
                if canon_nxt not in visited:
                    visited.add(canon_nxt)
                    orbit.append(nxt)
                    queue.append(nxt)
        return orbit

    # Integrable weights setup
    Q_dual = RootSystem(ct_aff.dual()).root_lattice()
    comarks = {i: Q_dual.null_root().coefficient(i) for i in index}

    int_weights_coeffs = []
    def find_weights(idx, current_sum, current_dict):
        if idx == rnk + 1:
            if current_sum == level:
                int_weights_coeffs.append(current_dict.copy())
            return
        i = index[idx]
        max_val = (level - current_sum) // comarks[i]
        for val in range(max_val + 1):
            current_dict[i] = val
            find_weights(idx + 1, current_sum + val * comarks[i], current_dict)
            del current_dict[i]

    find_weights(0, 0, {})
    int_weights = [sum(Lambda[i] * int_weights_coeffs[j][i] for i in index) for j in range(len(int_weights_coeffs))]

    # Character Assembly
    char_dict = {}
    for iwght in int_weights:
        int_rep = IntegrableRepresentation(iwght)
        md_weights = int_rep.dominant_maximal_weights()

        char = P(0).add_bigoh(max_ord)
        for wght in md_weights:
            v_dom = wght.to_classical().to_vector()
            mod_char = int_rep.modular_characteristic(wght)
            str_func = int_rep.string(wght, depth=max_ord)
            
            theta_orbit_sum = P(0).add_bigoh(max_ord)
            for v_orb in get_classical_orbit(v_dom):
                theta_orbit_sum += Theta(v_orb)

            char += (q**mod_char) * P(str_func) * theta_orbit_sum

        char_dict[tuple( iwght.to_vector()[:-1] )] = char
        #char_dict[iwght] = char

    return char_dict

In [1]:
%%cython
from sage.rings.power_series_ring import PowerSeriesRing
from sage.rings.puiseux_series_ring import PuiseuxSeriesRing
from sage.rings.rational_field import QQ
from sage.rings.integer_ring import ZZ

from sage.matrix.constructor import matrix
from sage.modules.free_module_element import vector

from sage.misc.functional import isqrt

import itertools as itr

'''
This block defines a function that takes two sets of characters and tries decomposes one into the other
It then does this decomposition for the diagonal coset characters being decomposed into the characters of the invariant algebra
'''

def char_decomp(list char1, list char2, int max_ord=10):
    '''
    Function to compute the decomposition of the characters in char1 into the characters in char2.
    max_ord is the order up to which the decomposition is checked
    A list of lists called solutions is returned.
    The number solutions[i][j] is the number of times character j in char2 appears in the decomposition of character i in char1
    '''

    # Power Series Ring the characters live in
    P = PuiseuxSeriesRing(QQ, 'q', sparse=True, default_prec=max_ord)

    # To speed things up, let us first check which decompositions are in principle possible 
    # according to the fractional part of the character exponents
    cdef list poss_decomps = [[] for _ in range(len(char1))]
    cdef int i,j
    cdef list val1 = [ QQ( char1[i].valuation() ) for i in range(len(char1)) ]
    cdef list val2 = [ QQ( char2[i].valuation() ) for i in range(len(char2)) ]
    for i,j in itr.product(range(len(char1)), range(len(char2))):
        if (val1[i]-val2[j]).denominator() == 1:
            poss_decomps[i] += [j]
    # print(poss_decomps)
            
    # Now we actually compute the decompositions
    cdef list expo
    cdef int num_char,max_ord_eff,n
    cdef int l,k
    # List of solutions
    cdef list solutions = [[] for _ in range(len(char1))]
    max_num = min([len(char.exponents())-1 for char in char2] + [len(char.exponents())-1 for char in char1])
    # print(max_num)
    for i in range(len(char1)):
        # To save time just skip ones where no decomposition is possible.
        if not poss_decomps[i]:
            continue
        # We now try solving it up to order 'n' and increase n until we fix a unique solution
        num_char = len(poss_decomps[i])
        char = char1[i]
        for n in range(num_char, max_num):
            expo = char.exponents()[:n]
            Vec = vector(ZZ, n)
            Mat = matrix(ZZ, num_char, n)
            for l in range(n):
                Vec[l] = char[expo[l]]
                for k in range(num_char):
                    Mat[k,l] = char2[poss_decomps[i][k]][expo[l]]
            # If the equation has no solution a ValueError is raised and we continue
            try:
                this_sol = Mat.solve_left(Vec)
            except ValueError:
                continue
            this_kern = Mat.kernel().basis()
            
            if len(this_kern)==0 or n == max_num-1:
                com_char = sum(P(this_sol[j]*char2[poss_decomps[i][j]], prec = max_ord) for j in range(num_char))
                if P(com_char-char, prec = max_ord-1) == 0:
                    solutions[i] += [ this_sol, poss_decomps[i] ]
                break

    # Return all the decompositions we have found
    return solutions


In [3]:
from coset_characters import compute_coset_characters
from affine_characters import affine_char_dict

# Now let's get the invariant algebra characters
inv_alg_char = load('invariant-algebra-characters')
inv_alg_char_list = list( inv_alg_char.values() )
inv_alg_char_ind = list( inv_alg_char.keys() )

# Some formatting helper functions

def format_coset_ind( coset_ind ):
    '''
    Function to format the indices of coset characters in latex
    '''
    ind_string = ''
    for ind in coset_ind:
        ind_string += str(ind) + ','
    return ind_string[:-1]

def format(tci):
    '''
    Function to format the indices used for the characters in latex
    '''
    if tci[1] == 'p':
        return fr'\chi_{{ {cdim[tci[0]]} }}^{{+}}'
    elif tci[1] == 'm':
        return fr'\chi_{{ {cdim[tci[0]]} }}^{{-}}'
    elif tci[1] == 'tp':
        return fr'\tilde{{ \chi }}_{{ {cdim[tci[0]]} }}^{{+}}'
    elif tci[1] == 'tm':
        return fr'\tilde{{ \chi }}_{{ {cdim[tci[0]]} }}^{{-}}'
    else:
        return fr'\chi_{{ {cdim[tci[0]] },{ cdim[tci[1]] } }}'

cdim = [0, 1/10, 3/5, 3/2, 3/80, 7/16] # Tricritical Ising squared conformal dimension

easy_cosets = load('easy_cosets') # Load a list of cosets

In [ ]:
# This is a check to make sure the characters aren't inconsitent with the modular data

import numpy as np

max_ord = 10 # Order to compute the character up to

all_chars = []

for coset in easy_cosets:
    print(coset[0],end='\n\n')

    num_chars = affine_char_dict(*coset[1], max_ord+2)
    den_chars_list = [ affine_char_dict(cartan_type, level, max_ord+2) for cartan_type,level in coset[2] ]
    weight_combos = coset[5]
    P_list = [ P for _,P in coset[3] ]

    # Actual characters
    coset_char = compute_coset_characters(num_chars, den_chars_list, weight_combos, P_list, max_ord=None, complete=True)

    combos = [tuple(tuple(w) for w in c) for c in coset[5]]
    chars  = [coset_char[c] for c in combos]  

    ev = lambda ch, t: sum(complex(CC(c)) * np.exp(2j*np.pi*t*float(e))
                       for e, c in zip(ch.exponents(), ch.coefficients()))
    S, T = coset[7]
    tau = 1.3j
    chi, chi1, chi2 = (np.array([ev(c, t) for c in chars]) for t in (tau, tau+1, -1/tau))
    Tn = np.array([complex(CC(T[i, i])) for i in range(len(combos))])
    Sn = np.array([[complex(CC(x)) for x in row] for row in S.rows()])
    print(abs(chi1 - Tn*chi).max(), abs(chi2 - Sn@chi).max(),end='\n\n')

C2_3 / A1_3 x A1_3

5.371689938816957e-15 1.3322676295501878e-15

D5_1 / A1_3 x A1_3

5.371689938816957e-15 3.885780586188048e-16

D7_1 / A1_1 x A1_1 x A1_3 x A1_3

5.371689938816957e-15 3.885780586188048e-16

D7_1 / A1_3 x A1_1 x A1_3 x A1_1

5.371689938816957e-15 4.163336342344337e-16

D7_1 / A1_3 x A1_1 x G2_1

5.371689938816957e-15 4.163336342344337e-16

D7_1 / G2_1 x A1_3 x A1_1

5.371689938816957e-15 4.163336342344337e-16

D7_1 / G2_1 x G2_1

5.371689938816957e-15 4.163336342344337e-16

D8_1 / A3_1 x A1_3 x A1_3

5.371689938816957e-15 3.885780586188048e-16

E8_1 / A3_1 x A1_3 x A1_3

5.371689938816957e-15 3.885780586188048e-16



[((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (0, 3), (1, 2)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (0, 3), (3, 0)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (1, 2), (0, 3)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (1, 2), (2, 1)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (2, 1), (1, 2)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (2, 1), (3, 0)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (3, 0), (0, 3)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (3, 0), (2, 1)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (0, 3), (0, 3)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (0, 3), (2, 1)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (1, 2), (1, 2)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (1, 2), (3, 0)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (2, 1), (0, 3)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (2, 1), (2, 1)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (3, 0), (1, 2)),
 ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (3, 0), (3, 0))]

In [ ]:
for coset in easy_cosets:
    print(coset[0],end='\n\n')

    print('Character decompositions in terms of invariant algebra:')

    num_chars = affine_char_dict(*coset[1], max_ord+2)
    den_chars_list = [ affine_char_dict(cartan_type, level, max_ord+2) for cartan_type,level in coset[2] ]
    weight_combos = coset[5]
    P_list = [ P for _,P in coset[3] ]

    # Actual characters
    coset_char = compute_coset_characters(num_chars, den_chars_list, weight_combos, P_list, max_ord=None, complete=True)

    coset_char_list = list( coset_char.values() )
    coset_char_ind = list( coset_char.keys() )
    all_chars.append( sorted(coset_char_list) )

    # No compute the decomposition
    our_decomps = char_decomp(coset_char_list, inv_alg_char_list,  max_ord = max_ord)

    for i in range(len(our_decomps)):
        if our_decomps[i]:
            print(rf'\chi_{{ {format_coset_ind(coset_char_ind[i])} }}',end=' &= ')
            facs,chars = our_decomps[i]
            for j in range(len(facs)):
                if j<len(facs)-1:
                    our_end = ' + '
                else:
                    our_end = '\n\n'
                print(rf'{'' if facs[j]==1 else str(facs[j])}{format(inv_alg_char_ind[chars[j]])}',end=our_end)

    
    print('Modular T-matrix:')
    print(coset[7][1])

    print('Modular S-matrix:')
    print(coset[7][0])

    print('')

In [4]:
fixed_all_chars = all_chars.copy()

NameError: name 'all_chars' is not defined

In [5]:
all_chars = fixed_all_chars.copy()

for i in range(len(all_chars)-1,-1,-1):
    these_chars = all_chars[i].copy()
    if these_chars in all_chars[:i]+all_chars[i+1:]:
        all_chars.pop(i)

In [66]:
char_DA = load('char_B_DA')
count = 0
seen_i = []
seen_j = []

for i,j in itr.product(range(39,-1,-1),repeat=2):
    test = char_DA[i] - all_chars[0][j]
    if test.add_bigoh(9) == 0:
        if i not in seen_i and j not in seen_j:
            seen_i.append(i)
            seen_j.append(j)
            count+=1

print(count)

36


In [78]:
char_cos

[q^(-7/120) + q^(233/120) + q^(353/120) + 3*q^(473/120) + 3*q^(593/120) + 7*q^(713/120) + 8*q^(833/120) + 15*q^(953/120) + 19*q^(1073/120) + 32*q^(1193/120) + 41*q^(1313/120) + O(q^(1433/120)),
 q^(-1/40) + q^(39/40) + 3*q^(79/40) + 6*q^(119/40) + 11*q^(159/40) + 18*q^(199/40) + 31*q^(239/40) + 49*q^(279/40) + 79*q^(319/40) + 120*q^(359/40) + 183*q^(399/40) + 271*q^(439/40) + O(q^(479/40)),
 1 + q + 2*q^2 + 4*q^3 + 7*q^4 + 12*q^5 + 20*q^6 + 32*q^7 + 50*q^8 + 77*q^9 + 116*q^10 + O(q^11),
 1 + q + 2*q^2 + 4*q^3 + 7*q^4 + 12*q^5 + 20*q^6 + 32*q^7 + 50*q^8 + 77*q^9 + 116*q^10 + O(q^11),
 q^(1/60) + 2*q^(61/60) + 5*q^(121/60) + 10*q^(181/60) + 18*q^(241/60) + 32*q^(301/60) + 53*q^(361/60) + 86*q^(421/60) + 136*q^(481/60) + 210*q^(541/60) + 318*q^(601/60) + 474*q^(661/60) + O(q^(721/60)),
 q^(1/24) + q^(25/24) + 3*q^(49/24) + 5*q^(73/24) + 10*q^(97/24) + 16*q^(121/24) + 28*q^(145/24) + 43*q^(169/24) + 70*q^(193/24) + 105*q^(217/24) + 161*q^(241/24) + 236*q^(265/24) + O(q^(289/24)),
 q^(1/24)

In [79]:
char_DA

[q^(-7/120) + q^(233/120) + q^(353/120) + 3*q^(473/120) + 3*q^(593/120) + 7*q^(713/120) + 8*q^(833/120) + 15*q^(953/120) + 19*q^(1073/120) + 32*q^(1193/120) + 41*q^(1313/120) + 66*q^(1433/120) + 86*q^(1553/120) + 130*q^(1673/120) + 173*q^(1793/120) + 250*q^(1913/120) + 331*q^(2033/120) + 469*q^(2153/120) + 619*q^(2273/120) + O(q^(319/16)),
 q^(-1/40) + q^(39/40) + 3*q^(79/40) + 6*q^(119/40) + 11*q^(159/40) + 18*q^(199/40) + 31*q^(239/40) + 49*q^(279/40) + 79*q^(319/40) + 120*q^(359/40) + 183*q^(399/40) + 271*q^(439/40) + 400*q^(479/40) + 576*q^(519/40) + 827*q^(559/40) + 1168*q^(599/40) + 1641*q^(639/40) + 2275*q^(679/40) + 3140*q^(719/40) + 4288*q^(759/40) + O(q^(319/16)),
 1 + 2*q + 4*q^2 + 8*q^3 + 14*q^4 + 24*q^5 + 40*q^6 + 64*q^7 + 100*q^8 + 154*q^9 + 232*q^10 + 344*q^11 + 504*q^12 + 728*q^13 + 1040*q^14 + 1470*q^15 + 2058*q^16 + 2856*q^17 + 3932*q^18 + 5372*q^19 + O(q^(2393/120)),
 1 + 2*q + 4*q^2 + 8*q^3 + 14*q^4 + 24*q^5 + 40*q^6 + 64*q^7 + 100*q^8 + 154*q^9 + 232*q^10 + 344*q^1

In [77]:
char_DA = sorted(char_DA, key= lambda x: x.valuation())
char_cos = sorted(all_chars[0], key= lambda x: x.valuation())

for i in range(40):
    if ( char_DA[i] - char_cos[i] ).add_bigoh(9) != 0:
        print(i)
        print( char_DA[i] - char_cos[i] )

2
q + 2*q^2 + 4*q^3 + 7*q^4 + 12*q^5 + 20*q^6 + 32*q^7 + 50*q^8 + 77*q^9 + 116*q^10 + O(q^11)
3
q + 2*q^2 + 4*q^3 + 7*q^4 + 12*q^5 + 20*q^6 + 32*q^7 + 50*q^8 + 77*q^9 + 116*q^10 + O(q^11)
4
1 - q^(1/60) + 2*q - 2*q^(61/60) + 4*q^2 - 5*q^(121/60) + 8*q^3 - 10*q^(181/60) + 14*q^4 - 18*q^(241/60) + 24*q^5 - 32*q^(301/60) + 40*q^6 - 53*q^(361/60) + 64*q^7 - 86*q^(421/60) + 100*q^8 - 136*q^(481/60) + 154*q^9 - 210*q^(541/60) + 232*q^10 - 318*q^(601/60) + 344*q^11 - 474*q^(661/60) + 504*q^12 + O(q^(721/60))
5
1 - q^(1/24) + 2*q - q^(25/24) + 4*q^2 - 3*q^(49/24) + 8*q^3 - 5*q^(73/24) + 14*q^4 - 10*q^(97/24) + 24*q^5 - 16*q^(121/24) + 40*q^6 - 28*q^(145/24) + 64*q^7 - 43*q^(169/24) + 100*q^8 - 70*q^(193/24) + 154*q^9 - 105*q^(217/24) + 232*q^10 - 161*q^(241/24) + 344*q^11 - 236*q^(265/24) + 504*q^12 + O(q^(289/24))
6
q^(1/60) - q^(1/24) + 2*q^(61/60) - q^(25/24) + 5*q^(121/60) - 3*q^(49/24) + 10*q^(181/60) - 5*q^(73/24) + 18*q^(241/60) - 10*q^(97/24) + 32*q^(301/60) - 16*q^(121/24) + 53*q^(361

In [36]:
for char_list in fixed_all_chars:
    print(len(char_list))

40
16
16
16
16
16
16
16
16


In [9]:
easy_cosets[4][:6]

['D7_1 / A1_3 x A1_1 x G2_1',
 ('D7', 1),
 [('A1', 3), ('A1', 1), ('G2', 1)],
 [('A1', [1 0 2 2 2 1 1]),
  ('A1', [1 2 2 2 2 1 1]),
  (
      [0 0 0 1 0 1 1]
'G2', [0 0 0 0 1 0 0]
)],
 2,
 [((0, 0, 0, 0, 0, 0, 0, 1), (2, 1), (0, 1), (0, 1, 0)),
  ((0, 0, 0, 0, 0, 0, 0, 1), (2, 1), (0, 1), (1, 0, 0)),
  ((0, 0, 0, 0, 0, 0, 0, 1), (3, 0), (1, 0), (0, 1, 0)),
  ((0, 0, 0, 0, 0, 0, 0, 1), (3, 0), (1, 0), (1, 0, 0)),
  ((0, 0, 0, 0, 0, 0, 1, 0), (2, 1), (0, 1), (0, 1, 0)),
  ((0, 0, 0, 0, 0, 0, 1, 0), (2, 1), (0, 1), (1, 0, 0)),
  ((0, 0, 0, 0, 0, 0, 1, 0), (3, 0), (1, 0), (0, 1, 0)),
  ((0, 0, 0, 0, 0, 0, 1, 0), (3, 0), (1, 0), (1, 0, 0)),
  ((0, 1, 0, 0, 0, 0, 0, 0), (2, 1), (0, 1), (0, 1, 0)),
  ((0, 1, 0, 0, 0, 0, 0, 0), (2, 1), (0, 1), (1, 0, 0)),
  ((0, 1, 0, 0, 0, 0, 0, 0), (3, 0), (1, 0), (0, 1, 0)),
  ((0, 1, 0, 0, 0, 0, 0, 0), (3, 0), (1, 0), (1, 0, 0)),
  ((1, 0, 0, 0, 0, 0, 0, 0), (2, 1), (0, 1), (0, 1, 0)),
  ((1, 0, 0, 0, 0, 0, 0, 0), (2, 1), (0, 1), (1, 0, 0)),
  ((1, 0, 0, 0

In [6]:
from coset_characters import compute_coset_characters
from affine_characters import affine_char_dict

easy_cosets = load('easy_cosets') # Load a list of cosets

# Now let's get the invariant algebra characters
inv_alg_char = load('invariant-algebra-characters')
inv_alg_char_list = list( inv_alg_char.values() )
inv_alg_char_ind = list( inv_alg_char.keys() )

max_ord = 10 # Order to compute the character up to
coset = easy_cosets[0]

print(coset[0],end='\n\n')

print('Character decompositions in terms of invariant algebra:')

num_chars = affine_char_dict(*coset[1], max_ord+2)
den_chars_list = [ affine_char_dict(cartan_type, level, max_ord+2) for cartan_type,level in coset[2] ]
weight_combos = coset[5]
P_list = [ P for _,P in coset[3] ]

# Actual characters
coset_char = compute_coset_characters(num_chars, den_chars_list, weight_combos, P_list, max_ord=None, complete=True)

coset_char_list = list( coset_char.values() )
coset_char_ind = list( coset_char.keys() )

# No compute the decomposition
our_decomps = char_decomp(coset_char_list, inv_alg_char_list,  max_ord = max_ord)

for i in range(len(our_decomps)):
    if our_decomps[i]:
        print(rf'\chi_{{ {format_coset_ind(coset_char_ind[i])} }}',end=' &= ')
        facs,chars = our_decomps[i]
        for j in range(len(facs)):
            if j<len(facs)-1:
                our_end = ' + '
            else:
                our_end = '\n\n'
            print(rf'{'' if facs[j]==1 else str(facs[j])}{format(inv_alg_char_ind[chars[j]])}',end=our_end)


print('Modular T-matrix:')
print(coset[7][1])

print('Modular S-matrix:')
print(coset[7][0])

C2_3 / A1_3 x A1_3

Character decompositions in terms of invariant algebra:
\chi_{ (1, 1, 1),(2, 1),(1, 2) } &= \chi_{ 3/80 }^{+} + \chi_{ 3/80 }^{-}

\chi_{ (1, 1, 1),(2, 1),(3, 0) } &= \chi_{ 7/16,3/80 }

\chi_{ (1, 1, 1),(3, 0),(0, 3) } &= \chi_{ 7/16 }^{+} + \chi_{ 7/16 }^{-}

\chi_{ (1, 1, 1),(3, 0),(2, 1) } &= \chi_{ 7/16,3/80 }

Modular T-matrix:
[   E(15)^4          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0]
[         0     E(3)^2          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0          0    

In [40]:
coset_char_list[39]

q^(-7/120) + q^(233/120) + q^(353/120) + 3*q^(473/120) + 3*q^(593/120) + 7*q^(713/120) + 8*q^(833/120) + 15*q^(953/120) + 19*q^(1073/120) + 32*q^(1193/120) + 41*q^(1313/120) + O(q^(1433/120))

In [7]:
# Here we define a function to bootstrap all possible modular invariants
def sym_commutant_basis(matrix_list):
    """
    Return a basis for the commutant of the matrices in matrix_list
    as a list of matrices.
    """
    if isinstance(matrix_list,tuple):
        matrix_list= list(matrix_list)
    if not isinstance(matrix_list,list):
        matrix_list = [matrix_list]
    
    # It is assumed thse are the same for all matrices in the list
    A0 = matrix_list[0]
    K = A0.base_ring()
    n = A0.nrows()

    Id = identity_matrix(K, n)
    Zero = matrix(K, n)
    # Start by enforcing symmetry
    M = identity_matrix(ZZ, n^2) - sum( matrix(ZZ, n, {(i,j):1}).tensor_product(matrix(ZZ, n, {(j,i):1})) for i,j in itr.product(range(n),repeat=2) )

    for A in matrix_list:
        # Build a matrix of linear constraints on the matrix entries
        M = M.stack(Id.tensor_product(A.transpose()) - A.tensor_product(Id))

    # Solve the constraints
    ker = M.right_kernel()

    # Convert vectors back into matrices
    basis = []
    for v in ker.basis():
        X = matrix(K, n, n, list(v))
        basis.append(X)

    return basis

In [ ]:
mod_inv_basis = sym_commutant_basis(easy_cosets[0][7])
save(mod_inv_basis,'coset0_mod_inv_basis')

In [66]:
for M in mod_inv_basis:
    print(M[39,39])

1/2
0
1/2
0
0
0


In [68]:
x = [var(f'x{i}') for i in range(6)]
test_M = sum(x[i]*mod_inv_basis[i] for i in range(6))

In [ ]:
x_val = [1,1,1,1,1,0] # Diagonal
#x_val = [1,1,1,1,0,0]
#x_val = [0,0,2,2,0,0]
spec_M = test_M

for i in range(len(x_val)):
    spec_M = spec_M.subs(x[i]==x_val[i])

print(spec_M)

[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0

In [64]:
entries = []
for i,j in itr.product(range(test_M.nrows()),repeat=2):
    if test_M[i,j] not in [0,1,2,3,4,5]+entries:
        entries.append(test_M[i,j])

In [65]:
entries

[x0,
 x1,
 x0 - x1,
 x2,
 x3,
 x2 - x3,
 1/2*x0 + 1/2*x2,
 -1/2*x0 + 1/2*x2,
 1/2*x1 + 1/2*x3,
 1/2*x0 - 1/2*x1 + 1/2*x2 - 1/2*x3,
 -1/2*x0 + 1/2*x1 + 1/2*x2 - 1/2*x3,
 -1/2*x1 + 1/2*x3,
 x4,
 x5,
 x0 - x1 - x5,
 x1 - x4,
 x0 - x1 + x4 - x5,
 x1 - x4 + x5]

In [85]:
S_matrix = [coset[7][0] for coset in easy_cosets[1:]]
T_matrix = [coset[7][1] for coset in easy_cosets[1:]]

for i in range(8):
    print(S_matrix[0]==S_matrix[i],T_matrix[0]==T_matrix[i],i)

True True 0
True True 1
False False 2
False False 3
False False 4
False False 5
True True 6
True True 7


In [94]:
S_matrix[0].is_similar(S_matrix[2],transformation=True)

(
      [                                                   1                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0                                                    0]
      [                                                   0                                                    1                              

In [97]:
C0 = S_matrix[0]**2
C2 = S_matrix[2]**2
perm = C0.is_similar(C2,transformation=True)[1]
perm*T_matrix[0]*perm == T_matrix[2]

False

In [106]:
T_matrix[0].diagonal()

[-E(12)^11,
 -E(60)^19,
 -E(12)^11,
 -E(60)^31,
 -E(60)^31,
 -E(12)^11,
 -E(60)^19,
 -E(12)^11,
 E(120)^53,
 E(24),
 -E(120)^77,
 -E(24),
 E(24),
 E(120)^77,
 -E(24),
 -E(120)^53]

In [112]:
all_chars = fixed_all_chars
vals = [ [ch.valuation() for ch in chars] for chars in all_chars ]
T_matrices = [coset[7][1] for coset in easy_cosets[:]]
T_diag = [ T.diagonal() for T in T_matrices ]

print(vals)
print(T_diag)

[[1, 1, 0, 0, 5/3, 5/3, 12/5, 12/5, 3/5, 3/5, 11/8, 11/8, 7/8, 7/8, 5/12, 5/12, 4/15, 1/15, 73/24, 73/24, 61/24, 61/24, 13/24, 13/24, 1/24, 1/24, 31/40, 19/40, 11/40, -1/40, 49/60, 1/60, 233/120, 173/120, 173/120, 137/120, 77/120, 77/120, 17/120, -7/120], [5/12, 5/12, 5/12, 5/12, 13/24, 13/24, 1/24, 1/24, 49/60, 49/60, 1/60, 1/60, 173/120, 77/120, 17/120, -7/120], [5/12, 5/12, 5/12, 5/12, 13/24, 13/24, 1/24, 1/24, 49/60, 49/60, 1/60, 1/60, 173/120, 77/120, 17/120, -7/120], [5/12, 5/12, 5/12, 5/12, 13/24, 13/24, 1/24, 1/24, 49/60, 49/60, 1/60, 1/60, 173/120, 77/120, 17/120, -7/120], [5/12, 5/12, 5/12, 5/12, 13/24, 13/24, 1/24, 1/24, 49/60, 49/60, 1/60, 1/60, 173/120, 77/120, 17/120, -7/120], [5/12, 5/12, 5/12, 5/12, 13/24, 13/24, 1/24, 1/24, 49/60, 49/60, 1/60, 1/60, 173/120, 77/120, 17/120, -7/120], [5/12, 5/12, 5/12, 5/12, 13/24, 13/24, 1/24, 1/24, 49/60, 49/60, 1/60, 1/60, 173/120, 77/120, 17/120, -7/120], [5/12, 5/12, 5/12, 5/12, 13/24, 13/24, 1/24, 1/24, 49/60, 49/60, 1/60, 1/60, 1

In [116]:
for coset in easy_cosets:
    print(coset[3])

[('A1', [1 1]), ('A1', [0 1])]
[('A1', [2 3 4 2 2]), ('A1', [0 1 0 1 1])]
[('A1', [1 0 0 0 0 0 0]), ('A1', [1 2 2 2 2 1 1]), ('A1', [0 0 2 3 4 2 2]), ('A1', [0 0 0 1 0 1 1])]
[('A1', [1 0 2 2 2 1 1]), ('A1', [1 2 2 2 2 1 1]), ('A1', [0 0 0 1 0 1 1]), ('A1', [0 0 0 1 2 1 1])]
[('A1', [1 0 2 2 2 1 1]), ('A1', [1 2 2 2 2 1 1]), ('G2', [0 0 0 1 0 1 1]
[0 0 0 0 1 0 0])]
[('G2', [1 0 2 2 2 1 1]
[0 1 0 0 0 0 0]), ('A1', [0 0 0 1 0 1 1]), ('A1', [0 0 0 1 2 1 1])]
[('G2', [1 0 2 2 2 1 1]
[0 1 0 0 0 0 0]), ('G2', [0 0 0 1 0 1 1]
[0 0 0 0 1 0 0])]
[('A3', [0 1 2 2 2 2 1 1]
[1 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0]), ('A1', [0 0 0 2 3 4 2 2]), ('A1', [0 0 0 0 1 0 1 1])]
[('A3', [ 0  1  1  2  2  2  2  1]
[-2 -3 -4 -6 -5 -4 -3 -2]
[ 0  0  0  0  0  0  0  1]), ('A1', [0 2 2 4 3 2 0 0]), ('A1', [0 1 1 0 1 0 0 0])]


In [ ]:
easy_cosets[0]

['E8_1 / A3_1 x A1_3 x A1_3',
 ('E8', 1),
 [('A3', 1), ('A1', 3), ('A1', 3)],
 [(
      [ 0  1  1  2  2  2  2  1]
      [-2 -3 -4 -6 -5 -4 -3 -2]
'A3', [ 0  0  0  0  0  0  0  1]
),
  ('A1', [0 2 2 4 3 2 0 0]),
  ('A1', [0 1 1 0 1 0 0 0])],
 2,
 [((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (0, 3), (1, 2)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (0, 3), (3, 0)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (1, 2), (0, 3)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (1, 2), (2, 1)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (2, 1), (1, 2)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (2, 1), (3, 0)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (3, 0), (0, 3)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (0, 1, 0, 0), (3, 0), (2, 1)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (0, 3), (0, 3)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (0, 3), (2, 1)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (1, 2), (1, 2)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (1, 2), (3, 0)),


In [37]:
easy_cosets[-2]

['D8_1 / A3_1 x A1_3 x A1_3',
 ('D8', 1),
 [('A3', 1), ('A1', 3), ('A1', 3)],
 [(
      [0 1 2 2 2 2 1 1]
      [1 0 0 0 0 0 0 0]
'A3', [0 1 0 0 0 0 0 0]
),
  ('A1', [0 0 0 2 3 4 2 2]),
  ('A1', [0 0 0 0 1 0 1 1])],
 4,
 [((0, 0, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0), (0, 3), (1, 2)),
  ((0, 0, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0), (0, 3), (3, 0)),
  ((0, 0, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0), (1, 2), (0, 3)),
  ((0, 0, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0), (1, 2), (2, 1)),
  ((0, 0, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0), (2, 1), (1, 2)),
  ((0, 0, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0), (2, 1), (3, 0)),
  ((0, 0, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0), (3, 0), (0, 3)),
  ((0, 0, 0, 0, 0, 0, 0, 1, 0), (0, 1, 0, 0), (3, 0), (2, 1)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (0, 3), (0, 3)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (0, 3), (2, 1)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (1, 2), (1, 2)),
  ((1, 0, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 0), (1, 2), (3, 0)),
  ((1, 0, 0, 0, 0, 0, 0,